# Reproducing the Explainable Deep Learning Framework

## Three-Stage Transfer Learning for Raman Spectroscopy-Based Bacterial Classification

<div class="alert alert-block alert-info">
<b>Note on Environment:</b> This notebook is optimized for execution within <b>Google Colab</b> (GPU runtime recommended). All file paths, dataset access patterns, and shell commands are structured for the Colab filesystem.
</div>

### 1. Abstract & Objectives

This document serves as the official, executable reproduction guide for the accompanying IEEE publication. It demonstrates the complete training and evaluation lifecycle of a Temporal Convolutional Network (TCN) designed to classify bacterial Raman spectra into clinically actionable antibiotic treatment groups.

By executing this notebook, you will automatically:
- **Provision the environment**: Mount datasets, clone the repository, and resolve dependencies.
- **Validate data integrity**: Execute `setup_data.py` to ensure correct loading and preprocessing of reference and clinical spectral splits.
- **Execute the Three-Stage Pipeline**:
  - **Stage 1**: Learn isolate-specific spectral features (30 classes).
  - **Stage 2**: Align feature representations to broad treatment categories (8 classes).
  - **Stage 3**: Transfer and evaluate on clinical patient spectra (5 classes) using 5-fold cross-validation.
- **Generate Interpretability Artifacts**: Run Local Interpretable Model-agnostic Explanations (LIME) and extract cross-architecture consensus Raman peaks.

### 2. Execution Workflow

The training pipeline executes the following sequence of scripts from the repository root:

| Order | Script | Objective |
| :--- | :--- | :--- |
| 1 | `scripts/setup_data.py` | Validates dataset presence and fits the spectral preprocessing pipeline. |
| 2 | `scripts/train.py --stage s1_isolate` | Executes Stage 1 (30-class reference isolate pretraining). |
| 3 | `scripts/train.py --stage s2_treatment` | Executes Stage 2 (8-class treatment pretraining). |
| 4 | `scripts/run_patient_cv.py --stage s3_transfer` | Orchestrates Stage 3 (5-fold patient-aware clinical transfer CV). |
| 5 | `scripts/analyze_experiment.py`| Aggregates cross-validation fold metrics. |
| 6 | `scripts/xai.py`               | Generates LIME and saliency feature attributions. |
| 7 | `scripts/compare_models_xai.py`| Extracts cross-architecture consensus Raman peaks. |


### 3. Environment Provisioning

#### 3.1. Hardware Verification
Confirm the presence of a hardware accelerator. While the models can be trained on a CPU, a GPU (e.g., NVIDIA T4) is strongly recommended to process the large reference datasets efficiently.

In [ ]:
import subprocess, sys

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
except FileNotFoundError:
    result = None

if result is not None and result.returncode == 0:
    print(f"GPU detected: {result.stdout.strip()}")
else:
    print("WARNING: No GPU detected. Training will be slow on CPU.")
    print("Recommended: Runtime > Change runtime type > T4 GPU")

print(f"Python version: {sys.version.split()[0]}")


#### 3.2. Storage Mount
Mount Google Drive to access the raw spectral datasets. The data must be located precisely at `/content/drive/MyDrive/Raman_spectral_classifier/data/raw/`.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive — a browser authentication prompt will appear
drive.mount('/content/drive')

# Verify the mount point
DRIVE_ROOT = '/content/drive/MyDrive/Raman_spectral_classifier'
DATA_RAW   = os.path.join(DRIVE_ROOT, 'data', 'raw')

print(f"Drive root : {DRIVE_ROOT}")
print(f"Data path  : {DATA_RAW}")
print(f"Path exists: {os.path.exists(DATA_RAW)}")

#### 3.3. Repository Clone & Data Linking
Clone the framework implementation into the active runtime. To avoid copying gigabytes of data, we create a symbolic link from the Drive mount to the repository's `data/raw/` directory.

In [ ]:
import os
import subprocess

REPO_DIR = "/content/raman-spectral-classifier"
REPO_URL = "https://github.com/rana-rohit/raman-spectral-classifier.git"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repository already cloned; pulling latest changes.")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
subprocess.run(["git", "log", "--oneline", "-5"], check=True)


In [ ]:
import os

# Symlink Drive data directory into the repo, avoiding large file copies
REPO_DATA = os.path.join(REPO_DIR, 'data', 'raw')
DRIVE_DATA = '/content/drive/MyDrive/Raman_spectral_classifier/data/raw'

# Remove existing symlink or empty directory if present
if os.path.islink(REPO_DATA):
    os.unlink(REPO_DATA)
elif os.path.isdir(REPO_DATA) and not os.listdir(REPO_DATA):
    os.rmdir(REPO_DATA)

if not os.path.exists(REPO_DATA):
    os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
    os.symlink(DRIVE_DATA, REPO_DATA)
    print(f"Symlinked: {REPO_DATA} -> {DRIVE_DATA}")
else:
    print(f"Data path already exists: {REPO_DATA}")

# Confirm symlink resolves
print(f"Resolved path: {os.path.realpath(REPO_DATA)}")
print(f"Path accessible: {os.path.exists(REPO_DATA)}")

#### 3.4. Dependency Resolution
Install required scientific libraries (`torch`, `scikit-learn`, `lime`, etc.) to satisfy `requirements.txt`.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

print()
print("Installation complete. Key library versions:")
import torch, numpy, scipy, sklearn
import importlib.metadata
print(f"  torch      : {torch.__version__}")
print(f"  numpy      : {numpy.__version__}")
print(f"  scipy      : {scipy.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  lime       : {importlib.metadata.version('lime')}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA device: {torch.cuda.get_device_name(0)}")


### 4. Dataset Organization

The framework consumes highly structured dataset partitions stored as serialized NumPy arrays. These arrays are mapped according to the label ontology defined in `metadata/ontology.py`.

| File | Split | Label Space | Description |
| :--- | :--- | :--- | :--- |
| `X_reference.npy` | Reference | Isolate (30-class) | Laboratory isolate spectra for pretraining |
| `X_test.npy` | Reference Test | Isolate (30-class) | Independent laboratory reference spectra |
| `X_finetune.npy` | Fine-tune | Treatment (5-class) | Small clinical adaptation set |
| `X_2018clinical.npy`| Clinical OOD | Treatment (5-class) | 2018 patient clinical spectra |
| `X_2019clinical.npy`| Clinical OOD | Treatment (5-class) | 2019 patient clinical spectra |
| `wavenumbers.npy` | Metadata | — | Wavenumber axis mapping (cm⁻¹) |

<div class="alert alert-block alert-warning">
<b>Dataset Availability:</b> Raw spectral datasets are not packaged within this repository. Ensure the files have been downloaded from the authoritative source (Ho et al., 2019) and placed in the <code>data/raw/</code> directory before proceeding.
</div>

### 5. Pipeline Integrity Validation

The `setup_data.py` module bootstraps the `DataRegistry`, verifies file presence, and fits the `SpectralPreprocessor` (e.g., Savitzky-Golay smoothing, SNV normalization) to the reference split. This must pass successfully to guarantee data integrity before training.

In [ ]:
import subprocess, sys

setup_commands = [
    [sys.executable, "scripts/setup_data.py", "--stage", "s1_isolate", "--split-mode", "iid_reference"],
    [sys.executable, "scripts/setup_data.py", "--stage", "s2_treatment", "--split-mode", "iid_reference"],
    [sys.executable, "scripts/setup_data.py", "--stage", "s3_transfer"],
]

for cmd in setup_commands:
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


> *Note: Outputs are generated dynamically upon execution. These checks validate that each stage can load its required splits and that the `SpectralPreprocessor` fits the reference distribution.*

### 6. Phase Execution: Three-Stage Transfer Learning

#### Stage 1: Isolate-Level Pretraining
We initialize the TCN architecture and train it to classify the 30 fine-grained bacterial/fungal reference isolates. This forces the model to learn a rich, discriminative spectral feature backbone. The resulting checkpoint becomes the foundation for Stage 2.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/raman_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_EXP_NAME = "tcn_s1_isolate_iid"
STAGE2_EXP_NAME = "tcn_s2_treatment_iid"
STAGE3_EXP_NAME = "tcn_s3_transfer_ts_iid_patient_cv"

STAGE1_EXP_DIR = OUTPUT_DIR / STAGE1_EXP_NAME
STAGE2_EXP_DIR = OUTPUT_DIR / STAGE2_EXP_NAME
STAGE3_EXP_DIR = OUTPUT_DIR / STAGE3_EXP_NAME

STAGE1_CKPT = STAGE1_EXP_DIR / "checkpoints" / "best_model.pt"
STAGE2_CKPT = STAGE2_EXP_DIR / "checkpoints" / "best_model.pt"
STAGE3_FOLD0_DIR = STAGE3_EXP_DIR / "fold_0_fold0"

print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/train.py",
        "--model", "tcn",
        "--stage", "s1_isolate",
        "--split-mode", "iid_reference",
        "--exp-name", STAGE1_EXP_NAME,
        "--exp-dir", str(OUTPUT_DIR),
        "--seed", "42",
    ],
    check=True,
)


> *Note: Running Stage 1 creates a runtime experiment directory containing checkpoints, logs, configuration snapshots, and metrics. The release repository ships the publication checkpoint under `artifacts/checkpoints/tcn/stage1_best.pt`.*

In [ ]:
if not STAGE1_CKPT.exists():
    raise FileNotFoundError(f"Stage 1 checkpoint not found: {STAGE1_CKPT}")
print(f"Stage 1 checkpoint ready: {STAGE1_CKPT}")


#### Stage 2: Treatment Representation Alignment
The Stage 1 feature backbone is loaded and frozen (or slowly fine-tuned), while a new classifier head is trained to predict 8 global antibiotic treatment groups. This stage logically bridges the gap between biological isolates and clinical treatment strategies.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/train.py",
        "--model", "tcn",
        "--stage", "s2_treatment",
        "--split-mode", "iid_reference",
        "--override",
        f"training.pretrained_checkpoint={STAGE1_CKPT}",
        "--exp-name", STAGE2_EXP_NAME,
        "--exp-dir", str(OUTPUT_DIR),
        "--seed", "42",
    ],
    check=True,
)


> *Note: Stage 2 loads the Stage 1 checkpoint from the active `OUTPUT_DIR`, matching the published workflow without embedding a user-specific absolute path.*

#### Stage 3: Clinical Patient Transfer
The Stage 2 treatment-aligned backbone is transferred to the sparse clinical environment (5 target treatment classes). Because clinical data is inherently grouped by patient, we enforce a strict 5-fold **patient-aware cross-validation** to prevent data leakage.

In [ ]:
if not STAGE2_CKPT.exists():
    raise FileNotFoundError(f"Stage 2 checkpoint not found: {STAGE2_CKPT}")

subprocess.run(
    [
        sys.executable,
        "scripts/run_patient_cv.py",
        "--model", "tcn",
        "--stage", "s3_transfer",
        "--exp-name", STAGE3_EXP_NAME,
        "--exp-dir", str(OUTPUT_DIR),
        "--seed", "42",
        "--override",
        f"training.pretrained_checkpoint={STAGE2_CKPT}",
    ],
    check=True,
)


> *Note: The 5-fold CV process yields five independent model checkpoints and aggregated predictions across all patients.*

### 7. Evaluation & Metrics Aggregation

The `analyze_experiment.py` script computes macroscopic performance metrics—including confusion matrices, precision/recall, and macro F1 scores—across the aggregated fold predictions.

In [ ]:
AGGREGATED_RESULTS = STAGE3_EXP_DIR / "aggregated_cv_results.json"
if not AGGREGATED_RESULTS.exists():
    raise FileNotFoundError(f"Aggregated Stage 3 results not found: {AGGREGATED_RESULTS}")
print(f"Aggregated Stage 3 results ready: {AGGREGATED_RESULTS}")


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/analyze_experiment.py",
        "--exp_dir", str(STAGE3_EXP_DIR),
        "--seed", "42",
    ],
    check=True,
)


### 8. Interpretability Pipeline

#### 8.1. Local Interpretable Explanations (LIME)
To validate that the TCN is utilizing biologically relevant Raman peaks rather than spurious dataset artifacts, we execute the LIME module to generate per-wavenumber feature attributions.

In [ ]:
if not STAGE3_FOLD0_DIR.exists():
    raise FileNotFoundError(f"Stage 3 fold directory not found: {STAGE3_FOLD0_DIR}")

subprocess.run(
    [
        sys.executable,
        "scripts/xai.py",
        "--exp-dir", str(STAGE3_FOLD0_DIR),
        "--seed", "42",
    ],
    check=True,
)


#### 8.2. Cross-Architecture Consensus Validation
We identify robust biomarker features by comparing the LIME attributions across multiple independent architectures (CNN, ResNet1D, Transformer, TCN) and finding consensus Raman peaks.

In [ ]:
REQUIRED_STAGE3_EXPERIMENTS = [
    "cnn_s3_transfer_ts_iid_patient_cv",
    "cnn_transformer_s3_transfer_ts_iid_patient_cv",
    "inception1d_s3_transfer_ts_iid_patient_cv",
    "resnet_s3_transfer_ts_iid_patient_cv",
    "tcn_s3_transfer_ts_iid_patient_cv",
    "transformer_s3_transfer_ts_iid_patient_cv",
]

missing_experiments = [
    name for name in REQUIRED_STAGE3_EXPERIMENTS
    if not (OUTPUT_DIR / name).is_dir()
]

if missing_experiments:
    print("Cross-architecture consensus requires all Stage 3 model runs.")
    print("Missing:", missing_experiments)
else:
    subprocess.run(
        [
            sys.executable,
            "scripts/compare_models_xai.py",
            "--results-root", str(OUTPUT_DIR),
            "--seed", "42",
        ],
        check=True,
    )


```text
artifacts/
|-- checkpoints/
|   |-- cnn/stage1_best.pt, stage2_best.pt
|   |-- cnn_transformer/stage1_best.pt, stage2_best.pt
|   |-- inception1d/stage1_best.pt, stage2_best.pt
|   |-- resnet1d/stage1_best.pt, stage2_best.pt
|   |-- tcn/stage1_best.pt, stage2_best.pt
|   `-- transformer/stage1_best.pt, stage2_best.pt
`-- results/
    |-- cnn/stage1/, stage2/
    |-- cnn_transformer/stage1/, stage2/
    |-- inception1d/stage1/, stage2/
    |-- resnet1d/stage2/
    |-- tcn/stage1/, stage2/
    `-- transformer/stage1/, stage2/
```
Stage 3 patient-CV outputs are created at runtime by the training pipeline and are not distributed with the repository.

#### Published Results Summary
| Stage | Metric | Value |
|-------|--------|-------|
| Stage 1 | Isolate classification accuracy | ~98.2% |
| Stage 2 | Treatment group accuracy | ~99.4% |
| Stage 3 | Clinical spectrum accuracy | **96.0%** |
| Stage 3 | **Patient-level accuracy** | **100.0%** |


### 10. Rigorous Reproducibility

To ensure experimental integrity, stochasticity is strictly controlled:
- **Global Random Seed**: Fixed at `--seed 42` for PyTorch, NumPy, and Python standard libraries.
- **Data Partitioning**: Contiguous allocation ensures consistent fold boundaries.
- **XAI Perturbation**: LIME utilizes `random_state=42` during surrogate sampling.

Minor numerical divergence (typically <0.1%) may still manifest across different hardware architectures due to non-deterministic CUDA atomic operations.

### 11. Concluding Remarks & Next Steps

You have successfully verified the repository environment, executed the three-stage transfer learning methodology, and generated the core results of the IEEE publication.

**Next Step:** Proceed to [Notebook 01 — Dataset Exploration](./01_dataset_exploration.ipynb) for a rigorous scientific analysis of the raw Raman spectra, the effects of Standard Normal Variate (SNV) normalization, and the logic underpinning the data augmentation pipeline.